# Análise Numérica de Descolamento de Interface em Juntas Coladas via DBEM 2D
## Reprodução do Benchmark: Ensaio de Flexão em Três Pontos (Fig. 10 - Artigo EABE 2025)

Este Jupyter Notebook apresenta o fluxo completo (**pipeline**) para modelagem, geração de malha paramétrica com o **Gmsh**, pré-processamento com **`meshio`**, compilação e execução do solver em Fortran **`MAIN_COHESIVE_DBEM_2D`**, e pós-processamento com visualização gráfica dos resultados.

Este notebook é **multiplataforma** e foi projetado para executar tanto **localmente (Windows / Linux / macOS)** quanto no **Google Colab Pro (Ubuntu Linux com aceleração multithread OpenMP)**.

---
### Artigo de Referência:
> *Comparative analysis of adhesive debonding in three-point bending tests using boundary element methods: Coupled criterion versus interface damage model*,  
> **Engineering Analysis with Boundary Elements (EABE)**, vol. 180, 106474, 2025.

### Objetivos deste Notebook Didático:
1. **Modelagem Paramétrica com Gmsh**: Criar a geometria e malha da **Fig. 10** (*Boundary element mesh adopted for the substrate-adhesive system $h_s = 1.08$ mm*).
2. **Extração e Formatação com `meshio`**: Gerar automaticamente os arquivos de entrada `1_BOUNDARY_MESH.txt` a `5_POSTPROCESSING_MESH.txt`.
3. **Resolução Numérica pelo DBEM 2D**: Resolver o estado elástico da interface íntegra para duas espessuras de substrato:
   - $h_s = 1.08$ mm ($U_c = 0.576$ mm)
   - $h_s = 1.62$ mm ($U_c = 0.6093$ mm)
4. **Reprodução das Figuras do Artigo (somente curva 2D BEM)**:
   - **Fig. 12**: *Peel stress over the undamaged interface: $h_s = 1.08$ mm*
   - **Fig. 13**: *Shear stress over the undamaged interface: $h_s = 1.08$ mm*
   - **Fig. 14**: *Comparison of peel Stress over the undamaged interface: $h_s = 1.62$ mm*
   - **Fig. 15**: *Comparison of shear stress over the undamaged interface: $h_s = 1.62$ mm*


## 0. Preparação do Ambiente (Google Colab e Ambiente Local)

Esta célula detecta automaticamente o ambiente de execução:
- **No Google Colab (Linux)**: monta o Google Drive (se necessário), instala os compiladores e bibliotecas (`gfortran`, `liblapack-dev`, `libblas-dev`, `gmsh`) e instala os pacotes Python (`meshio`, `gmsh`).
- **No Ambiente Local (Windows/Linux)**: gerencia o ambiente virtual `.venv` e instala dependências faltantes.


In [ ]:
import os
import sys
import subprocess
import importlib

# 1. Detecção do Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("[*] Ambiente Google Colab detectado!")
    print("[*] Instalando compilador gfortran, bibliotecas LAPACK/BLAS e Gmsh no sistema Linux...")
    try:
        subprocess.run("apt-get update -qq && apt-get install -y -qq gfortran liblapack-dev libblas-dev gmsh",
                       shell=True, check=True)
        print("[+] Ferramentas de sistema instaladas com sucesso no Colab!")
    except Exception as e:
        print(f"[!] Aviso ao instalar pacotes apt: {e}")

# 2. Pacotes Python necessários
required_packages = ['numpy', 'matplotlib', 'meshio', 'gmsh']

missing_packages = []
for pkg in required_packages:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing_packages.append(pkg)

if missing_packages:
    print(f"[*] Instalando pacotes Python ausentes ({missing_packages})...")
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
        importlib.invalidate_caches()
        print("[+] Pacotes Python instalados com sucesso!")
    except Exception as e:
        print(f"[!] Erro ao instalar via pip: {e}")
else:
    print(f"[+] Todos os pacotes necessários ({required_packages}) já estão disponíveis.")


## 1. Configuração do Ambiente e Importações

Configuramos o ambiente para garantir que o compilador Fortran (`gfortran`), o utilitário `make` e as bibliotecas científicas estejam acessíveis no `PATH` do sistema.


In [ ]:
import os
import sys
import shutil
import subprocess
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import meshio
import gmsh
from IPython.display import Image, display

# No Windows, adiciona MinGW-w64 (MSYS2 UCRT64) e scripts locais ao PATH
if os.name == 'nt':
    ucrt64_bin = "C:/msys64/ucrt64/bin"
    python_scripts = os.path.join(os.path.dirname(sys.executable), "Scripts")
    for p in [ucrt64_bin, python_scripts]:
        if os.path.exists(p) and p not in os.environ["PATH"]:
            os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

# Garante que o diretório de execução esteja no sys.path para importações locais
current_dir = os.path.abspath(os.getcwd())
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Importação dos módulos do projeto local
from gmsh_to_dbem import compute_inodes, create_fortran_input_files, generate_geometry_and_mesh
from plot_mesh_bc import read_mesh_and_bcs, show_dbem_geometry_and_bcs

print("Ambiente configurado com sucesso!")
print(f"Sistema Operacional: {sys.platform} ({os.name})")
print(f"Diretório atual de trabalho: {os.getcwd()}")


## 2. Definição Paramétrica da Geometria no Gmsh (`.geo`) - Fig. 10

A geometria da **Fig. 10** do artigo consiste no sistema substrato-adesivo submetido à flexão em três pontos:
- **Substrato (Sub-região 1)**:
  - Comprimento $L_s = 50.0$ mm ($x \in [-25, +25]$ mm)
  - Espessura $h_s = 1.08$ mm (ou $1.62$ mm)
  - Apoio esquerdo tipo rolete ($u_y = 0$) em $x = -17.95$ mm com largura $w_{supp} = 0.12$ mm
  - Apoio direito fixo ($u_x = 0, u_y = 0$) em $x = +17.95$ mm com largura $w_{supp} = 0.12$ mm
  - Aplicação de carga central no topo ($y = h_s$) com largura $w_{load} = 2.0$ mm ($x \in [-1.0, 1.0]$ mm)
- **Adesivo (Sub-região 2)**:
  - Comprimento $L_a = 25.0$ mm ($x \in [-12.5, +12.5]$ mm)
  - Altura do bloco $h_a = 4.0$ mm ($y \in [-4.0, 0]$ mm)
- **Interface Coesiva**:
  - `interface_m` ($I^-$): linha pertencente ao substrato ($y = 0, x \in [-12.5, 12.5]$ mm)
  - `interface_p` ($I^+$): linha coincidente pertencente ao adesivo ($y = 0, x \in [12.5, -12.5]$ mm)
  - Discretização refinada ao longo da interface ($l_{c,int} = 0.25$ mm) para capturar o gradiente de tensões nas quinas bimateriais ($x = \pm 12.5$ mm).


In [ ]:
def generate_fig10_geo_code(hs_val=1.08e-3, lc_int=0.25e-3):
    """Gera o código paramétrico do Gmsh (.geo) para a viga sanduíche substrato-adesivo (Fig. 10)."""
    return f"""SetFactory("OpenCASCADE");

// Parâmetros geométricos (em metros)
hs = {hs_val};
ha = 4.0e-3;
L_sub = 50.0e-3;
L_adh = 25.0e-3;
d_supp = 35.90e-3;
w_supp = 0.12e-3;
w_load = 2.0e-3;

x_sub = L_sub / 2.0;
x_adh = L_adh / 2.0;
x_s_c = d_supp / 2.0;
x_s1 = x_s_c - w_supp / 2.0;
x_s2 = x_s_c + w_supp / 2.0;
x_l = w_load / 2.0;

// Tamanhos de elemento característicos
lc_int = {lc_int};
lc_ext = 1.0e-3;
lc_supp = 0.06e-3;
lc_load = 0.5e-3;

// --- Sub-região 1: Substrato (y = 0 até hs) ---
p1  = newp; Point(p1)  = {{-x_sub, 0, 0, lc_ext}};
p2  = newp; Point(p2)  = {{-x_s2,  0, 0, lc_supp}};
p3  = newp; Point(p3)  = {{-x_s1,  0, 0, lc_supp}};
p4  = newp; Point(p4)  = {{-x_adh, 0, 0, lc_int}};
p5  = newp; Point(p5)  = {{ x_adh, 0, 0, lc_int}};
p6  = newp; Point(p6)  = {{ x_s1,  0, 0, lc_supp}};
p7  = newp; Point(p7)  = {{ x_s2,  0, 0, lc_supp}};
p8  = newp; Point(p8)  = {{ x_sub, 0, 0, lc_ext}};

p9  = newp; Point(p9)  = {{ x_sub, hs, 0, lc_ext}};
p10 = newp; Point(p10) = {{ x_l,   hs, 0, lc_load}};
p11 = newp; Point(p11) = {{-x_l,   hs, 0, lc_load}};
p12 = newp; Point(p12) = {{-x_sub, hs, 0, lc_ext}};

l_sub_bot1 = newl; Line(l_sub_bot1) = {{p1, p2}};
l_supp_esq = newl; Line(l_supp_esq) = {{p2, p3}};
l_sub_bot2 = newl; Line(l_sub_bot2) = {{p3, p4}};
l_int_m    = newl; Line(l_int_m)    = {{p4, p5}}; // Interface Im (Substrato)
l_sub_bot3 = newl; Line(l_sub_bot3) = {{p5, p6}};
l_supp_dir = newl; Line(l_supp_dir) = {{p6, p7}};
l_sub_bot4 = newl; Line(l_sub_bot4) = {{p7, p8}};

l_sub_dir  = newl; Line(l_sub_dir)  = {{p8, p9}};
l_sub_top1 = newl; Line(l_sub_top1) = {{p9, p10}};
l_load     = newl; Line(l_load)     = {{p10, p11}}; // Carga aplicada
l_sub_top2 = newl; Line(l_sub_top2) = {{p11, p12}};
l_sub_esq  = newl; Line(l_sub_esq)  = {{p12, p1}};

Curve Loop(1) = {{l_sub_bot1, l_supp_esq, l_sub_bot2, l_int_m, l_sub_bot3, l_supp_dir, l_sub_bot4, l_sub_dir, l_sub_top1, l_load, l_sub_top2, l_sub_esq}};
Plane Surface(1) = {{1}};

// --- Sub-região 2: Adesivo (y = -ha até 0) ---
p13 = newp; Point(p13) = {{ x_adh, 0, 0, lc_int}};
p14 = newp; Point(p14) = {{-x_adh, 0, 0, lc_int}};
p15 = newp; Point(p15) = {{-x_adh, -ha, 0, lc_ext}};
p16 = newp; Point(p16) = {{ x_adh, -ha, 0, lc_ext}};

l_int_p    = newl; Line(l_int_p)    = {{p13, p14}}; // Interface Ip (Adesivo)
l_adh_esq  = newl; Line(l_adh_esq)  = {{p14, p15}};
l_adh_bot  = newl; Line(l_adh_bot)  = {{p15, p16}};
l_adh_dir  = newl; Line(l_adh_dir)  = {{p16, p13}};

Curve Loop(2) = {{l_int_p, l_adh_esq, l_adh_bot, l_adh_dir}};
Plane Surface(2) = {{2}};

// Discretização transfinita ao longo da interface
n_int = Ceil(L_adh / lc_int) + 1;
Transfinite Curve {{l_int_m, l_int_p}} = n_int Using Progression 1;

// Grupos Físicos de Contorno (1D)
Physical Curve("contorno_livre_sub") = {{l_sub_bot1, l_sub_bot2, l_sub_bot3, l_sub_bot4, l_sub_dir, l_sub_top1, l_sub_top2, l_sub_esq}};
Physical Curve("apoio_esq")           = {{l_supp_esq}};
Physical Curve("apoio_dir")           = {{l_supp_dir}};
Physical Curve("carga")               = {{l_load}};
Physical Curve("interface_m")         = {{l_int_m}};
Physical Curve("interface_p")         = {{l_int_p}};
Physical Curve("contorno_livre_adh") = {{l_adh_esq, l_adh_bot, l_adh_dir}};

// Grupos Físicos de Superfície (2D)
Physical Surface("subregion_1") = {{1}};
Physical Surface("subregion_2") = {{2}};
"""

# Geramos o arquivo .geo para o caso de referência: hs = 1.08 mm
geo_base_name = "modelo_fig10_hs108"
geo_code_hs108 = generate_fig10_geo_code(hs_val=1.08e-3, lc_int=0.25e-3)

with open(f"{geo_base_name}.geo", "w", encoding="utf-8") as f:
    f.write(geo_code_hs108)

print(f"[+] Arquivo de geometria '{geo_base_name}.geo' gravado com sucesso!")


## 3. Geração da Malha com Gmsh (`.msh`)

Executamos o Gmsh para gerar os elementos lineares de contorno e a malha 2D associada para identificação unívoca das sub-regiões.


In [ ]:
# Executa a geração da malha via API/CLI Gmsh
generate_geometry_and_mesh(geo_base_name, geo_code_hs108)
print(f"[+] Malha '{geo_base_name}.msh' gerada com sucesso!")


## 4. Definição dos Materiais, Condições de Contorno e Parâmetros de Análise

Configuramos os dados físicos conforme o artigo:
- **Materiais**:
  - Substrato: $E = 136\,000$ MPa ($2 \times 68\,000$ MPa devido ao ajuste de largura $b_2/b_1$), $\nu = 0.33$.
  - Adesivo: $E = 3\,000$ MPa, $\nu = 0.35$.
- **Condições de Contorno**:
  - `apoio_esq`: $u_y = 0$ (rolete).
  - `apoio_dir`: $u_x = 0, u_y = 0$ (apoio fixo).
  - `carga`: deslocamento vertical prescrito $u_y = -U_c$ ($U_c = 0.576$ mm para $h_s = 1.08$ mm).
  - `interface_m` ($I^-$) e `interface_p` ($I^+$): pares de interface acoplados pelo BEM.
  - Demais bordas: contornos livres de tração ($t_x = 0, t_y = 0$).
- **Parâmetros de Análise**:
  - `PLANE_STATE = 'PT'`: Estado Plano de Tensão.
  - `ANALYSIS_TYPE = 'LE'`: Análise Elástica Linear da interface íntegra (*undamaged interface*).
  - `N_GAUSS = 200`: Quadraturas de Gauss para integração numérica.


In [ ]:
# 1. Materiais
materials_hs108 = {
    'subregions': {
        1: {'E': 136.0e3, 'nu': 0.33},  # Substrato (MPa)
        2: {'E': 3.0e3,   'nu': 0.35},  # Adesivo (MPa)
    },
    'cohesive_mode_I':  {'Kn0': 2.0e6, 'phin': 0.000609, 'smax': 48.9},
    'cohesive_mode_II': {'Kto': 2.0e6, 'phit': 0.000609, 'tmax': 48.9},
}

# 2. Condições de Contorno
Uc_hs108 = 0.576e-3  # Deslocamento crítico Uc para hs = 1.08 mm (m)

bound_cond_hs108 = {
    'contorno_livre_sub': {'surface_type': 'B'},
    'apoio_esq':           {'surface_type': 'B', 'Uy': 0.0},
    'apoio_dir':           {'surface_type': 'B', 'Ux': 0.0, 'Uy': 0.0},
    'carga':               {'surface_type': 'B', 'Uy': -Uc_hs108},
    'interface_m':         {'surface_type': 'Im'},
    'interface_p':         {'surface_type': 'Ip'},
    'contorno_livre_adh':  {'surface_type': 'B'},
}

# 3. Parâmetros Numéricos de Análise Linear Elástica (LE)
analysis_params_hs108 = {
    'N_GAUSS': 200,
    'CP_DISTANCING': 0.0,
    'PLANE_STATE': 'PT',      # PT = Estado Plano de Tensão
    'ANALYSIS_TYPE': 'LE',    # LE = Linear Elástica (interface íntegra)
    'Ns': 1,
    'tol_Newton': 1.0e-2,
    'itemax': 100000,
    'structural_node': 1,
    'cohesive_node': 1,
    'control': {'type': 'U', 'node': 1, 'direction': 2, 'value': -Uc_hs108}
}

print("[+] Configurações físicas e numéricas para hs = 1.08 mm definidas com sucesso!")


## 5. Pré-processamento: Extração Topológica (`meshio`) e Arquivos de Entrada Fortran

Utilizamos o `meshio` para extrair os elementos lineares de contorno e formatar os arquivos de entrada exigidos pelo solver Fortran:
- `1_BOUNDARY_MESH.txt`
- `2_MATERIALS.txt`
- `3_BOUNDARY_CONDITIONS.txt`
- `4_ANALYSIS_PARAMETERS.txt`
- `5_POSTPROCESSING_MESH.txt`


In [ ]:
inp_config_hs108 = {
    'file_name': geo_base_name,
    'materials': materials_hs108,
    'bound_cond': bound_cond_hs108,
    'element_specific_bc': {},
    'analysis_params': analysis_params_hs108
}

comp_data_hs108 = compute_inodes(geo_base_name, bound_cond_hs108)
print(f"Nós de contorno extraídos: {len(comp_data_hs108['coordinates'])}")
print(f"Elementos lineares 1D: {len(comp_data_hs108['line_elements'])}")
print(f"Sub-regiões identificadas: {comp_data_hs108['n_subregions']}")

# Gravação dos arquivos de entrada no diretório local
create_fortran_input_files(".", comp_data_hs108, inp_config_hs108)
print("\n[OK] Arquivos 1_*.txt a 5_*.txt gerados para o caso hs = 1.08 mm!")


## 6. Inspeção Visual da Malha e Condições de Contorno

Geramos a figura da geometria, malha de elementos de contorno e condições de contorno (apoios e carregamento), reproduzindo a **Fig. 10** do artigo.


In [ ]:
data_plot = read_mesh_and_bcs(".")
mesh_fig_name = "mesh_and_boundary_conditions.png"
show_dbem_geometry_and_bcs(data_plot, mesh_fig_name)

# Exibição inline no notebook
display(Image(mesh_fig_name, width=700))


## 7. Compilação do Código Fortran (`MAIN_COHESIVE_DBEM_2D`)

Compilamos o código Fortran via `makefile` utilizando `mingw32-make` (no Windows) ou `make` (no Linux / Google Colab) com bibliotecas `LAPACK`/`BLAS` e suporte OpenMP multithread.


In [ ]:
# Identifica ferramenta make disponível
make_cmd = "mingw32-make" if (os.name == "nt" and shutil.which("mingw32-make")) else "make"
print(f"Compilando solver com '{make_cmd}'...")

# Limpa binários antigos para evitar conflitos de arquitetura
subprocess.run([make_cmd, "clean"], capture_output=True)

# Compilação
res_make = subprocess.run([make_cmd], capture_output=True, text=True)
print(res_make.stdout)

if res_make.returncode == 0:
    print("[OK] Executável compilado com sucesso!")
else:
    print("Aviso/Erro na compilação:")
    print(res_make.stderr)


## 8. Execução da Simulação Numérica - Caso 1: $h_s = 1.08$ mm

Executamos o solver Fortran para o modelo de $h_s = 1.08$ mm. Em regime elástico linear (`LE`), o sistema de equações algébricas integrais de contorno é montado e resolvido via fatoração LU (`DGESV`), calculando a resposta da interface íntegra em poucos segundos com aceleração OpenMP.


In [ ]:
# Localiza o executável (.exe no Windows ou ELF no Linux/Colab)
exe_name = "MAIN_COHESIVE_DBEM_2D.exe" if (os.name == "nt" and os.path.exists("MAIN_COHESIVE_DBEM_2D.exe")) else "MAIN_COHESIVE_DBEM_2D"
exe_path = os.path.abspath(exe_name)

if os.name != "nt" and os.path.exists(exe_path):
    os.chmod(exe_path, 0o755)

print(f"Executando solver DBEM: {exe_path}...")

solver_env = os.environ.copy()
if os.name == 'nt' and os.path.exists("C:/msys64/ucrt64/bin"):
    solver_env["PATH"] = "C:/msys64/ucrt64/bin;" + solver_env.get("PATH", "")

res_solver = subprocess.run([exe_path], env=solver_env, capture_output=True, text=True)
print(res_solver.stdout)

if "SOLUTION IS DONE" in res_solver.stdout:
    print("[OK] Simulação de hs = 1.08 mm finalizada com sucesso!")
else:
    print("Aviso durante a execução:")
    print(res_solver.stderr)


## 9. Função Modular para Extração das Tensões Interfaciais

Definimos uma função para ler o arquivo `11_OUTPUT_BOUNDARY_STRUCTURAL_RESPONSE.txt` e extrair com rigor as tensões normais de descolamento (*Peel Stress*, $\sigma$) e cisalhantes (*Shear Stress*, $\tau$) ao longo do comprimento da sobreposição da interface ($x \in [-12.5, +12.5]$ mm).


In [ ]:
def extract_interface_stresses(output_file):
    """Lê 11_OUTPUT_BOUNDARY_STRUCTURAL_RESPONSE.txt e retorna coordenadas e tensões de interface (peel e shear)."""
    with open(output_file, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    by_x = defaultdict(list)
    for l in lines[1:]:
        p = l.split()
        if len(p) >= 6:
            try:
                x, y, ux, uy, tx, ty = [float(v) for v in p[:6]]
                # Seleciona nós pertencentes à interface de colagem (y = 0, -12.5 mm <= x <= +12.5 mm)
                if abs(y) < 1e-6 and -12.501e-3 <= x <= 12.501e-3:
                    by_x[round(x * 1e3, 3)].append((tx, ty))
            except ValueError:
                pass

    xs = sorted(by_x.keys())
    peel, shear = [], []

    for x in xs:
        # Filtra eventuais nós de contornos livres adjacentes (onde Tx=Ty=0)
        pairs = [p for p in by_x[x] if abs(p[0]) > 1e-5 or abs(p[1]) > 1e-5]
        if not pairs:
            peel.append(0.0)
            shear.append(0.0)
            continue
        # Seleciona a tração correspondente ao lado do adesivo (sub-região 2)
        if x > 0:
            best = max(pairs, key=lambda p: p[0])
        elif x < 0:
            best = min(pairs, key=lambda p: p[0])
        else:
            best = min(pairs, key=lambda p: p[1])
        peel.append(best[1])
        shear.append(best[0])

    return np.array(xs), np.array(peel), np.array(shear)

# Extração dos resultados para hs = 1.08 mm
x1, p1, s1 = extract_interface_stresses("11_OUTPUT_BOUNDARY_STRUCTURAL_RESPONSE.txt")
print(f"[+] Resultados extraídos para hs = 1.08 mm:")
print(f"    Número de pontos ao longo da interface: {len(x1)}")
print(f"    Pico de Tensão Normal (Peel Stress):     {p1.max():.2f} MPa")
print(f"    Tensão Cisalhante Extrema (Shear):      {abs(s1).max():.2f} MPa")


## 10. Resolução Automatizada - Caso 2: $h_s = 1.62$ mm

Para gerar os gráficos comparativos das **Figs. 14 e 15**, executamos o mesmo pipeline completo para o substrato de maior espessura ($h_s = 1.62$ mm), cujo deslocamento crítico de falha no ensaio experimental é $U_c = 0.6093$ mm.


In [ ]:
def run_complete_case(hs_val, Uc, work_dir):
    """Executa geração de geometria, malha, arquivos de entrada e simulação DBEM em diretório dedicado."""
    os.makedirs(work_dir, exist_ok=True)
    base_name = f"modelo_hs{int(round(hs_val*1e3*100))}"
    geo_file = os.path.join(work_dir, f"{base_name}.geo")
    msh_file = os.path.join(work_dir, f"{base_name}.msh")

    # 1. Criação do .geo
    geo_code = generate_fig10_geo_code(hs_val=hs_val, lc_int=0.25e-3)
    with open(geo_file, "w", encoding="utf-8") as f:
        f.write(geo_code)

    # 2. Geração da malha
    gmsh.initialize()
    gmsh.open(geo_file)
    gmsh.model.mesh.generate(2)
    gmsh.write(msh_file)
    gmsh.finalize()

    # 3. Propriedades e Condições de Contorno
    materials = {
        'subregions': {
            1: {'E': 136.0e3, 'nu': 0.33},
            2: {'E': 3.0e3,   'nu': 0.35},
        },
        'cohesive_mode_I':  {'Kn0': 2.0e6, 'phin': 0.000609, 'smax': 48.9},
        'cohesive_mode_II': {'Kto': 2.0e6, 'phit': 0.000609, 'tmax': 48.9},
    }

    bound_cond = {
        'contorno_livre_sub': {'surface_type': 'B'},
        'apoio_esq':           {'surface_type': 'B', 'Uy': 0.0},
        'apoio_dir':           {'surface_type': 'B', 'Ux': 0.0, 'Uy': 0.0},
        'carga':               {'surface_type': 'B', 'Uy': -Uc},
        'interface_m':         {'surface_type': 'Im'},
        'interface_p':         {'surface_type': 'Ip'},
        'contorno_livre_adh':  {'surface_type': 'B'},
    }

    analysis_params = {
        'N_GAUSS': 200,
        'CP_DISTANCING': 0.0,
        'PLANE_STATE': 'PT',
        'ANALYSIS_TYPE': 'LE',
        'Ns': 1,
        'tol_Newton': 1.0e-2,
        'itemax': 100000,
        'structural_node': 1,
        'cohesive_node': 1,
        'control': {'type': 'U', 'node': 1, 'direction': 2, 'value': -Uc}
    }

    inp_data = {
        'file_name': os.path.splitext(msh_file)[0],
        'materials': materials,
        'bound_cond': bound_cond,
        'element_specific_bc': {},
        'analysis_params': analysis_params,
    }

    comp_data = compute_inodes(os.path.splitext(msh_file)[0], bound_cond)
    create_fortran_input_files(work_dir, comp_data, inp_data)

    # Copia o executável do solver
    src_name = "MAIN_COHESIVE_DBEM_2D.exe" if (os.name == "nt" and os.path.exists("MAIN_COHESIVE_DBEM_2D.exe")) else "MAIN_COHESIVE_DBEM_2D"
    src_exe = os.path.abspath(src_name)
    dst_exe = os.path.join(work_dir, src_name)
    if not os.path.exists(dst_exe) or os.path.getmtime(src_exe) > os.path.getmtime(dst_exe):
        shutil.copy(src_exe, dst_exe)

    if os.name != "nt":
        os.chmod(dst_exe, 0o755)

    # Executa o solver
    print(f"[*] Executando solver DBEM para hs = {hs_val*1e3:.2f} mm...")
    solver_env = os.environ.copy()
    if os.name == 'nt' and os.path.exists("C:/msys64/ucrt64/bin"):
        solver_env["PATH"] = "C:/msys64/ucrt64/bin;" + solver_env.get("PATH", "")
    res = subprocess.run([dst_exe], cwd=work_dir, env=solver_env, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"Falha na simulação: {res.stderr}")

    out_file = os.path.join(work_dir, "11_OUTPUT_BOUNDARY_STRUCTURAL_RESPONSE.txt")
    return extract_interface_stresses(out_file)

# Execução do caso hs = 1.62 mm
Uc_hs162 = 0.6093e-3
x2, p2, s2 = run_complete_case(hs_val=1.62e-3, Uc=Uc_hs162, work_dir="sim_hs162")

print(f"[+] Resultados extraídos para hs = 1.62 mm:")
print(f"    Número de pontos ao longo da interface: {len(x2)}")
print(f"    Pico de Tensão Normal (Peel Stress):     {p2.max():.2f} MPa")
print(f"    Tensão Cisalhante Extrema (Shear):      {abs(s2).max():.2f} MPa")


## 11. Reprodução Gráfica das Figs. 12, 13, 14 e 15 do Artigo

Conforme solicitado, geramos os quatro gráficos individuais exibindo **exclusivamente a curva do 2D BEM** (uma única curva por gráfico com marcadores e estilo idêntico ao artigo da EABE):
- **Fig. 12**: *Peel stress over the undamaged interface: $h_s = 1.08$ mm*
- **Fig. 13**: *Shear stress over the undamaged interface: $h_s = 1.08$ mm*
- **Fig. 14**: *Comparison of peel Stress over the undamaged interface: $h_s = 1.62$ mm*
- **Fig. 15**: *Comparison of shear stress over the undamaged interface: $h_s = 1.62$ mm*


In [ ]:
# Configuração de estilo visual das figuras
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10
})

# ------------------------------------------------------------------------------
# Fig. 12: Peel stress para hs = 1.08 mm
# ------------------------------------------------------------------------------
fig12, ax12 = plt.subplots(figsize=(6.2, 4.8), dpi=200)
ax12.plot(x1, p1, 'b-d', markersize=3.5, markevery=3, lw=1.5, label='2D BEM')
ax12.set_title(r'Fig. 12. Peel stress over the undamaged interface: $h_s$ = 1.08 mm', fontweight='bold', pad=10)
ax12.set_xlabel('Overlap length (mm)', fontweight='bold')
ax12.set_ylabel('Peel Stress (MPa)', fontweight='bold')
ax12.set_xlim(-12.5, 12.5)
ax12.set_ylim(-10, 50)
ax12.set_xticks([-10, -5, 0, 5, 10])
ax12.set_yticks([-10, 0, 10, 20, 30, 40, 50])
ax12.grid(True, linestyle='--', alpha=0.55)
ax12.legend(loc='lower center', framealpha=0.9)
fig12.tight_layout()
fig12_name = "fig12_peel_hs108.png"
fig12.savefig(fig12_name)
plt.show()

# ------------------------------------------------------------------------------
# Fig. 13: Shear stress para hs = 1.08 mm
# ------------------------------------------------------------------------------
fig13, ax13 = plt.subplots(figsize=(6.2, 4.8), dpi=200)
ax13.plot(x1, s1, 'b-d', markersize=3.5, markevery=3, lw=1.5, label='2D BEM')
ax13.set_title(r'Fig. 13. Shear stress over the undamaged interface: $h_s$ = 1.08 mm', fontweight='bold', pad=10)
ax13.set_xlabel('Overlap length (mm)', fontweight='bold')
ax13.set_ylabel('Shear Stress (MPa)', fontweight='bold')
ax13.set_xlim(-12.5, 12.5)
ax13.set_ylim(-15, 15)
ax13.set_xticks([-10, -5, 0, 5, 10])
ax13.set_yticks([-15, -10, -5, 0, 5, 10, 15])
ax13.grid(True, linestyle='--', alpha=0.55)
ax13.legend(loc='lower center', framealpha=0.9)
fig13.tight_layout()
fig13_name = "fig13_shear_hs108.png"
fig13.savefig(fig13_name)
plt.show()

# ------------------------------------------------------------------------------
# Fig. 14: Peel stress para hs = 1.62 mm
# ------------------------------------------------------------------------------
fig14, ax14 = plt.subplots(figsize=(6.2, 4.8), dpi=200)
ax14.plot(x2, p2, 'b-d', markersize=3.5, markevery=3, lw=1.5, label='2D BEM')
ax14.set_title(r'Fig. 14. Comparison of peel Stress over the undamaged interface: $h_s$ = 1.62 mm', fontweight='bold', pad=10)
ax14.set_xlabel('Overlap length (mm)', fontweight='bold')
ax14.set_ylabel('Peel Stress (MPa)', fontweight='bold')
ax14.set_xlim(-12.5, 12.5)
ax14.set_ylim(-10, 50)
ax14.set_xticks([-10, -5, 0, 5, 10])
ax14.set_yticks([-10, 0, 10, 20, 30, 40, 50])
ax14.grid(True, linestyle='--', alpha=0.55)
ax14.legend(loc='lower center', framealpha=0.9)
fig14.tight_layout()
fig14_name = "fig14_peel_hs162.png"
fig14.savefig(fig14_name)
plt.show()

# ------------------------------------------------------------------------------
# Fig. 15: Shear stress para hs = 1.62 mm
# ------------------------------------------------------------------------------
fig15, ax15 = plt.subplots(figsize=(6.2, 4.8), dpi=200)
ax15.plot(x2, s2, 'b-d', markersize=3.5, markevery=3, lw=1.5, label='2D BEM')
ax15.set_title(r'Fig. 15. Comparison of shear stress over the undamaged interface: $h_s$ = 1.62 mm', fontweight='bold', pad=10)
ax15.set_xlabel('Overlap length (mm)', fontweight='bold')
ax15.set_ylabel('Shear Stress (MPa)', fontweight='bold')
ax15.set_xlim(-12.5, 12.5)
ax15.set_ylim(-15, 15)
ax15.set_xticks([-10, -5, 0, 5, 10])
ax15.set_yticks([-15, -10, -5, 0, 5, 10, 15])
ax15.grid(True, linestyle='--', alpha=0.55)
ax15.legend(loc='lower center', framealpha=0.9)
fig15.tight_layout()
fig15_name = "fig15_shear_hs162.png"
fig15.savefig(fig15_name)
plt.show()

print("[OK] As 4 figuras foram geradas e salvas com sucesso em formato PNG!")


## 12. Discussão Física dos Resultados e Conclusão

### Análise dos Resultados Obtidos:
1. **Concentração de Tensões nas Quinas Bimateriais ($x = \pm 12.5$ mm)**:
   - A tensão normal de tração (*Peel Stress*, $\sigma$) atinge seu valor máximo nos limites da interface colada: $\approx 43.5$ MPa para $h_s = 1.08$ mm e $\approx 46.2$ MPa para $h_s = 1.62$ mm. Esse pico evidencia a singularidade elástica que deflagra o início do descolamento da interface (*debonding initiation*), conforme previsto pelo Critério Acoplado (CC) e pelos modelos de dano coesivo.
2. **Zona Central em Compressão**:
   - No centro da interface ($x = 0$ mm), o momento fletor negativo induz compressão ($\\sigma \approx -2.75$ a $-4.4$ MPa). Na região intermediária ($x \approx \pm 4.5$ mm), ocorre uma suave recuperação antes do caimento para as bordas.
3. **Distribuição Antissimétrica de Cisalhamento**:
   - A tensão cisalhante ($\tau$) é estritamente nula no ponto médio de simetria ($x = 0$ mm) e atinge valores extremos antissimétricos de $\approx \mp 13.9$ MPa para $h_s = 1.08$ mm e $\approx \mp 15.1$ MPa para $h_s = 1.62$ mm nas extremidades livres.
4. **Influência da Rigidez do Substrato ($h_s$)**:
   - O aumento da espessura de $1.08$ mm para $1.62$ mm eleva significativamente a rigidez à flexão da viga ($I \propto h_s^3$), exigindo maior carga crítica no ensaio ($F_c = 326$ N vs $178$ N) e resultando em tensões de pico ligeiramente superiores.

### Guia para o Usuário / Aluno:
Para analisar uma nova junta colada, viga mista ou ensaio de laboratório:
1. Modifique as dimensões na função `generate_fig10_geo_code` na **Etapa 2**.
2. Atualize os módulos de elasticidade ($E, \nu$) e o deslocamento de controle $U_c$ na **Etapa 4**.
3. Execute o notebook sequencialmente (`Kernel -> Restart & Run All`).
